## 05-supervisor.ipynb

멀티 에이전트 패턴 중 하나
- 도메인이 여러개 섞여 있고
- 각 도메인마다 도구가 많고 복잡함.
- 하위 담당 에이전트와 사용자가 소통할 필요가 없음
- 단순 도구만 활용할 경우에는 사용X

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model('gpt-4.1-mini')

In [ ]:
from langchain.tools import tool

@tool
def create_calender_event(
    title : str,
    start_time: str,
    end_time: str,
    attendees: list[str],   # ['a@a.com','b@b.com']
    location: str = '',     # 위치가 없을 경우, 빈 문자열. 디폴트셋팅
):
    '''캘린더 이벤트 생성'''
    return f'이벤트 생성 완료 {title} - {start_time} ~ {end_time}'

@tool
def send_email(
    to: list[str],
    subject: str,
    body: str,
    attendees: list[str], 
):
    '''이메일 발송'''
    return f'이메일 발송 완료. {to} - {subject}'


@tool
def get_available_time_slot(
    attendees: list[str], 
    date: str,
    duration_minutes: int
):
    '''참가자들이 특정 날짜에 참여 가능한 시간 확인'''
    
    return ['09:00', '14:00', '16:00']


In [8]:
from langchain.agents import create_agent
from datetime import datetime

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
    f"NOW: {datetime.now()}"
)

calendar_agent = create_agent(
    model,
    tools=[create_calender_event, get_available_time_slot],
    system_prompt=CALENDAR_AGENT_PROMPT
)


In [13]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

In [14]:
query = '다음주 화요일 오전 10시에 1시간동안 팀 미팅을 잡아줘'

for step in email_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]}
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_kTUxRtGPXJZ5RBgIhRIcmTDO)
 Call ID: call_kTUxRtGPXJZ5RBgIhRIcmTDO
  Args:
    to: ['team@company.com']
    subject: 팀 미팅 일정 공지: 다음주 화요일 오전 10시
    body: 안녕하세요 팀원 여러분,

다음주 화요일 오전 10시부터 1시간 동안 팀 미팅을 진행하고자 합니다. 자세한 일정과 안건은 추후 공유드리겠습니다.

참석 부탁드립니다.

감사합니다.
    attendees: ['team@company.com']
================================= Tool Message =================================
Name: send_email

이메일 발송 완료. ['team@company.com'] - 팀 미팅 일정 공지: 다음주 화요일 오전 10시
================================== Ai Message ==================================

다음주 화요일 오전 10시에 1시간 동안 진행되는 팀 미팅 일정 공지를 팀원들에게 이메일로 발송했습니다. 다른 도움이 필요하시면 말씀해 주세요.


In [17]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

In [18]:
SUPERVISOR_PROMPT = '''너는 매우 유능한 개인 비서야.
너는 캘린더 이벤트를 조정하고, 이메일을 보낼 수 있어.
사용자 요청을 분석해서 적절한 도구를 사용하고 결과를 종합해야해.
요청이 여러가지 액션을 취해야하면 순서를 잘 짜서 각종 도구들을 여러번 호출해
'''

suervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    )

In [22]:
query = '내일 오전 9시에 팀 아침 회의 잡아줘'

for step in suervisor_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]}
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_ZMKMKE8l6LDIgXM4BQ1mTyNA)
 Call ID: call_ZMKMKE8l6LDIgXM4BQ1mTyNA
  Args:
    request: 팀 아침 회의 내일 오전 9시
================================= Tool Message =================================
Name: schedule_event

팀 아침 회의를 내일 오전 9시부터 9시 30분까지 일정으로 예약했습니다. 다른 도움이 필요하신가요?
================================== Ai Message ==================================

팀 아침 회의를 내일 오전 9시부터 9시 30분까지 예약했습니다. 다른 일정이나 도움이 필요하시면 알려주세요.
